[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C43_Data_Engineering_Course/01_dedup_scale/01_dedup_scale.ipynb)

# 01 · 大规模去重（用 numpy/标准库从零实现 + 对拍 + 算账）

目标：把 **精确去重 → MinHash 估 Jaccard → LSH banding → SimHash 汉明 → 跨分片并查集合并** 从零实现，每个工程化版本都**对拍**暴力参考、每个机制都**算一笔复杂度账**。

路线：shingle 表示 → 暴力 Jaccard(参考) → MinHash 无偏性 → LSH 分桶 + S 曲线 → SimHash 汉明 → 跨分片合并 → ✏️ 练习 → 📖 答案 → 🧪 FineWeb 去重胶囊。

> 心智模型：**MinHash = 把集合压成定长签名；LSH = 只把可能相似的拉到同桶比；并查集 = 把传递的重复归成簇**。我们写算法*结构*与*正确性*，规模由账目推演。

## 1 · shingle 表示 + 暴力 Jaccard（建立「真值」参考）

把文档切成 **k-shingle**（连续 k 个词的集合），用 Jaccard 衡量相似度。
暴力两两 Jaccard 是 **O(n²)** 的，但它是我们后面所有近似方法要对拍的**真值参考**。

In [ ]:
import numpy as np, hashlib, re
from collections import defaultdict
rng = np.random.default_rng(0)

def shingles(text, k=3):
    '''返回文档的 k-词-shingle 集合。'''
    words = re.findall(r'\w+', text.lower())
    if len(words) < k:
        return {' '.join(words)} if words else set()
    return {' '.join(words[i:i+k]) for i in range(len(words) - k + 1)}

def jaccard(a, b):
    if not a and not b: return 1.0
    return len(a & b) / len(a | b)

d1 = 'the quick brown fox jumps over the lazy dog'
d2 = 'the quick brown fox jumps over the lazy cat'   # 近重复：只改最后一词
d3 = 'completely different sentence about machine learning systems'
s1, s2, s3 = shingles(d1), shingles(d2), shingles(d3)
print(f'J(d1,d2) = {jaccard(s1,s2):.3f}  <- 近重复，高')
print(f'J(d1,d3) = {jaccard(s1,s3):.3f}  <- 无关，低')
assert jaccard(s1, s2) > 0.5, '改一个词应仍高度相似'
assert jaccard(s1, s3) < 0.1, '无关文档应几乎不相似'
assert jaccard(s1, s1) == 1.0
print('✅ shingle + Jaccard 正确：这是后面所有近似方法的「真值」裁判')

## 2 · MinHash 估 Jaccard（验证无偏性）

用 `K` 个独立哈希，对集合取每个哈希下的**最小值**，得到 K 维签名。
**对应位相等的比例 = Jaccard 的无偏估计**，方差 ≈ J(1−J)/K。

我们用 `K` 组随机 `(a,b)` 做 `(a*x+b) mod p` 模拟独立哈希族，再验证估计值随 K 增大逼近真 Jaccard。

In [ ]:
MERSENNE = (1 << 61) - 1     # 一个大素数 p

def make_hashes(K, seed=0):
    '''K 组随机 (a,b)，定义 h_i(x) = (a_i*x + b_i) mod p。'''
    r = np.random.default_rng(seed)
    a = r.integers(1, MERSENNE, size=K)
    b = r.integers(0, MERSENNE, size=K)
    return a, b

def minhash_signature(shingle_set, a, b):
    '''对集合算 K 维 MinHash 签名。'''
    if not shingle_set:
        return np.full(len(a), MERSENNE, dtype=np.int64)
    # 把每个 shingle 映射成一个 64-bit 整数
    xs = np.array([int(hashlib.sha1(s.encode()).hexdigest()[:15], 16) for s in shingle_set], dtype=np.int64)
    # 对每个哈希函数取最小：形状 (n_shingle, K) -> min over axis 0
    H = (np.outer(xs, a) + b) % MERSENNE        # (n_shingle, K)
    return H.min(axis=0)

def minhash_estimate(sigA, sigB):
    return float(np.mean(sigA == sigB))

true_J = jaccard(s1, s2)
print(f'真 Jaccard(d1,d2) = {true_J:.3f}')
print(f"{'K':>6s} {'MinHash 估计':>14s} {'|误差|':>10s}")
for K in [16, 64, 256, 1024]:
    a, b = make_hashes(K, seed=1)
    est = minhash_estimate(minhash_signature(s1, a, b), minhash_signature(s2, a, b))
    print(f'{K:>6d} {est:>14.3f} {abs(est-true_J):>10.3f}')
# K 越大，估计越接近真值（方差 ~ J(1-J)/K）
a, b = make_hashes(1024, seed=1)
est_big = minhash_estimate(minhash_signature(s1, a, b), minhash_signature(s2, a, b))
assert abs(est_big - true_J) < 0.08, 'K=1024 时估计应很接近真 Jaccard'
print('\n✅ MinHash 是 Jaccard 的无偏估计：K 越大越准。一篇文档被压成 K 维定长签名。')

## 3 · LSH banding：分桶 + S 形概率曲线

把 K 维签名切成 `b` 个 band、每 band `r` 行（K=b·r）。每个 band 打包哈希成桶 id；
两文档**至少一个 band 完全相同**即成候选。命中概率 `P(J) = 1 − (1 − J^r)^b` 是陡峭的 **S 曲线**，
近似阈值 `τ ≈ (1/b)^(1/r)`。先验证这条曲线，再用它分桶。

In [ ]:
def lsh_hit_prob(J, b, r):
    return 1.0 - (1.0 - J**r) ** b

def lsh_threshold(b, r):
    return (1.0 / b) ** (1.0 / r)

b, r = 20, 5      # K = 100
tau = lsh_threshold(b, r)
print(f'b={b}, r={r} -> K={b*r}, 近似阈值 τ ≈ {tau:.3f}')
print(f"{'Jaccard':>8s} {'命中概率':>10s}")
for J in [0.3, 0.5, tau, 0.8, 0.95]:
    print(f'{J:>8.3f} {lsh_hit_prob(J, b, r):>10.3f}')
# S 曲线性质：阈值以下概率低、以上概率高
assert lsh_hit_prob(0.3, b, r) < 0.2, '远低于阈值的几乎不该成候选'
assert lsh_hit_prob(0.95, b, r) > 0.9, '远高于阈值的几乎必成候选'
assert lsh_hit_prob(0.95, b, r) > lsh_hit_prob(0.5, b, r)  # 单调
print('✅ S 曲线正确：调 (b,r) 即调阈值；这是 LSH 区分相似/不相似的核心')

现在用 banding **真正分桶**，把「两两比对」换成「只比同桶的」，并对拍暴力 Jaccard 的召回。

In [ ]:
def lsh_candidates(signatures, b, r):
    '''signatures: (n, K) 矩阵。返回候选对集合 {(i,j)}。'''
    n, K = signatures.shape
    assert K == b * r
    candidates = set()
    for band in range(b):
        buckets = defaultdict(list)
        sub = signatures[:, band*r:(band+1)*r]
        for i in range(n):
            key = hash(sub[i].tobytes())     # 该 band 的 r 个值打包成桶 id
            buckets[key].append(i)
        for idxs in buckets.values():
            for x in range(len(idxs)):
                for y in range(x+1, len(idxs)):
                    candidates.add((idxs[x], idxs[y]))
    return candidates

# 造一个小语料：每篇随机选一个「主题模板」+ 少量改动 -> 内含近重复簇
templates = ['machine learning models train on large text corpora and learn patterns',
             'the weather today is sunny with a gentle breeze and clear skies',
             'distributed systems must handle failures replication and consistency']
corpus = []
for _ in range(60):
    t = templates[rng.integers(0, len(templates))]
    words = t.split()
    if rng.random() < 0.6:                  # 60% 概率改 1~2 个词 -> 近重复
        for _ in range(rng.integers(1, 3)):
            words[rng.integers(0, len(words))] = f'w{rng.integers(0,99)}'
    corpus.append(' '.join(words))

sh = [shingles(d, k=3) for d in corpus]
a, b_h = make_hashes(b*r, seed=2)
sigs = np.stack([minhash_signature(s, a, b_h) for s in sh])
cands = lsh_candidates(sigs, b, r)

# 暴力真值：所有 J>=0.7 的对
TAU = 0.7
truth = {(i,j) for i in range(len(sh)) for j in range(i+1,len(sh)) if jaccard(sh[i],sh[j])>=TAU}
found = {(i,j) for (i,j) in cands if jaccard(sh[min(i,j)],sh[max(i,j)])>=TAU}
recall = len(found & truth) / max(len(truth), 1)
print(f'暴力需比 {len(sh)*(len(sh)-1)//2} 对；LSH 只生成 {len(cands)} 个候选对')
print(f'真重复对 {len(truth)} 个，LSH 找回 {len(found & truth)} 个 -> 召回 {recall:.0%}')
assert recall >= 0.9, 'LSH 召回应该很高（漏检少）'
assert len(cands) < len(sh)*(len(sh)-1)//2, 'LSH 候选数应远小于暴力对数'
print('✅ LSH 用远少的候选对抓回几乎所有真重复 —— O(n²) 变近 O(n) 的核心')

## 4 · 完整 MinHash+LSH 去重管线（对拍暴力去重结果）

把前面拼成一条管线：MinHash 签名 → LSH 候选 → 真 Jaccard 精筛 → 并查集归簇 → 每簇留一份。
**与「暴力两两 Jaccard 去重」对拍**：保留的文档集合应一致（或召回≥某阈值）。

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]    # 路径压缩
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[max(ra,rb)] = min(ra,rb)   # 总把大根并到小根 -> 代表=最小下标

def dedup_minhash_lsh(sh_sets, b, r, tau, seed=3):
    n = len(sh_sets)
    a, bb = make_hashes(b*r, seed=seed)
    sigs = np.stack([minhash_signature(s, a, bb) for s in sh_sets])
    cands = lsh_candidates(sigs, b, r)
    uf = UnionFind(n)
    for (i, j) in cands:
        if jaccard(sh_sets[i], sh_sets[j]) >= tau:     # 精筛去假阳
            uf.union(i, j)
    keep = sorted({uf.find(i) for i in range(n)})      # 每簇留代表(最小下标)
    return keep

def dedup_brute(sh_sets, tau):
    '''暴力 O(n^2) 去重参考：并查集 + 两两真 Jaccard。'''
    n = len(sh_sets); uf = UnionFind(n)
    for i in range(n):
        for j in range(i+1, n):
            if jaccard(sh_sets[i], sh_sets[j]) >= tau:
                uf.union(i, j)
    return sorted({uf.find(i) for i in range(n)})

keep_lsh = dedup_minhash_lsh(sh, b, r, TAU)
keep_brute = dedup_brute(sh, TAU)
print(f'原始 {len(sh)} 篇 -> 暴力去重保留 {len(keep_brute)} 篇，MinHash+LSH 保留 {len(keep_lsh)} 篇')
# 两者保留的簇数应该非常接近（LSH 可能因偶发漏检多留极少数）
assert abs(len(keep_lsh) - len(keep_brute)) <= 2, 'LSH 去重结果应与暴力高度一致'
print('✅ MinHash+LSH 管线与暴力 O(n²) 去重结果一致 —— 工程版正确')

## 5 · SimHash：b 位指纹 + 汉明距离

另一条路线：把每个特征哈希成 `B` 位、按权投票得到 `B` 位**指纹**；相似文档指纹的**汉明距离小**。
签名仅 B bit（比 MinHash 省），比对是一次异或数 1。

In [ ]:
def simhash(shingle_set, B=64):
    '''返回 B 位指纹(int)。每个 shingle 哈希成 B 位，按位 +1/-1 投票。'''
    v = np.zeros(B, dtype=np.int64)
    for s in shingle_set:
        h = int(hashlib.sha1(s.encode()).hexdigest(), 16)
        for bit in range(B):
            v[bit] += 1 if (h >> bit) & 1 else -1
    fp = 0
    for bit in range(B):
        if v[bit] > 0: fp |= (1 << bit)
    return fp

def hamming(x, y):
    return bin(x ^ y).count('1')

f1, f2, f3 = simhash(s1), simhash(s2), simhash(s3)
print(f'汉明(d1,d2) = {hamming(f1,f2):>2d}  <- 近重复，小')
print(f'汉明(d1,d3) = {hamming(f1,f3):>2d}  <- 无关，大')
assert hamming(f1, f2) < hamming(f1, f3), '近重复的汉明距离应更小'
assert hamming(f1, f1) == 0
print('✅ SimHash 正确：指纹仅 64 bit，汉明距离一条指令算完 —— 比 MinHash 省空间')

## 6 · 跨分片合并：分布式去重的最后一步

PB 级语料分成多个 shard 在不同机器上处理。**跨 shard 的重复**要靠把候选对汇总 + 全局并查集归簇。
验证：把语料切成多个 shard、各自 LSH 出候选，再**全局合并**，结果应与「全量一起去重」一致。

In [ ]:
def shard_candidates(sh_sets, shard_ids, b, r, seed=3):
    '''模拟分布式：按 band 桶 id 把 *所有 shard* 的签名汇到一起出候选
       （这正是 MapReduce shuffle：相同桶 id 汇到同一 reducer，跨 shard 也能配上）。'''
    a, bb = make_hashes(b*r, seed=seed)
    sigs = np.stack([minhash_signature(s, a, bb) for s in sh_sets])
    return lsh_candidates(sigs, b, r)    # 桶按 band 全局建 -> 自然跨 shard

n = len(sh)
shard_ids = [i % 4 for i in range(n)]      # 把文档轮流分到 4 个 shard
cands_global = shard_candidates(sh, shard_ids, b, r)
uf = UnionFind(n)
cross = 0
for (i, j) in cands_global:
    if jaccard(sh[i], sh[j]) >= TAU:
        if shard_ids[i] != shard_ids[j]: cross += 1
        uf.union(i, j)
keep_sharded = sorted({uf.find(i) for i in range(n)})
print(f'4 个 shard；其中 {cross} 个真重复对是跨 shard 的（单机 shard 内去重会漏掉它们！）')
print(f'全局合并后保留 {len(keep_sharded)} 篇；全量去重保留 {len(keep_brute)} 篇')
assert abs(len(keep_sharded) - len(keep_brute)) <= 2, '跨分片合并应等价于全量去重'
assert cross > 0, '应当存在跨 shard 的重复（否则示例没意义）'
print('✅ 跨分片合并正确：并查集把跨 shard 的重复也归到同一簇 —— 分布式去重的关键')

---
## ✏️ 练习 1：从零实现 MinHash 签名（向量化）

实现 `my_minhash(shingle_set, a, b, p)`：给定哈希族参数 `(a,b)`（各 K 维）与素数 `p`，
返回该集合的 K 维 MinHash 签名（每个哈希取所有 shingle 的最小值）。空集返回全 `p`。

提示：把 shingle 先映射成整数数组 `xs`，再对每个哈希 `(a_i*x+b_i)%p` 取最小。

In [ ]:
def my_minhash(shingle_set, a, b, p=MERSENNE):
    K = len(a)
    # TODO: 空集 -> np.full(K, p)
    #       否则把每个 shingle 用 sha1 映射成 int 数组 xs，
    #       计算 (outer(xs,a)+b) % p，对 axis=0 取 min，返回 K 维签名
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
a, b = make_hashes(128, seed=7)
mine = my_minhash(s1, a, b)
ref  = minhash_signature(s1, a, b)
assert mine.shape == (128,)
assert np.array_equal(mine, ref), '应与参考 MinHash 实现逐位一致'
# 空集 -> 全 p
assert np.all(my_minhash(set(), a, b) == MERSENNE)
# 自相似 = 1
assert minhash_estimate(my_minhash(s1,a,b), my_minhash(s1,a,b)) == 1.0
print('✅ 练习 1 通过：MinHash 签名实现正确')

## ✏️ 练习 2：给定目标阈值反解 (b, r)

去重要把 LSH 阈值卡在目标 Jaccard 附近。实现 `pick_b_r(K, target_tau)`：
在所有满足 `b*r == K` 的 `(b,r)` 中（r 从 1 到 K，b=K//r 且整除），
选近似阈值 `(1/b)^(1/r)` **最接近** `target_tau` 的一组，返回 `(b, r, 实际阈值)`。

In [ ]:
def pick_b_r(K, target_tau):
    # TODO: 遍历 r in 1..K，若 K % r == 0 则 b=K//r，算 tau=(1/b)**(1/r)，
    #       记录 |tau - target_tau| 最小的 (b, r, tau)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
b2, r2, tau2 = pick_b_r(100, 0.8)
assert b2 * r2 == 100, 'b*r 必须等于 K'
assert abs(tau2 - 0.8) < 0.12, '实际阈值应接近目标 0.8'
# 目标阈值更高 -> 应选更大的 r（更苛刻）
b_lo, r_lo, _ = pick_b_r(100, 0.5)
b_hi, r_hi, _ = pick_b_r(100, 0.9)
assert r_hi >= r_lo, '更高的目标阈值通常对应更大的 r'
print(f'K=100, 目标 0.8 -> b={b2}, r={r2}, 实际阈值={tau2:.3f}')
print('✅ 练习 2 通过：能据目标阈值反解 (b,r)')

## ✏️ 练习 3：SimHash 汉明去重

实现 `simhash_dedup(sh_sets, B, max_ham)`：对每篇算 B 位 SimHash 指纹，
用并查集把**汉明距离 ≤ max_ham** 的文档归簇，返回每簇代表（最小下标，排序）。

（这是 SimHash 版的去重，可与第 4 节 MinHash 版对比。）

In [ ]:
def simhash_dedup(sh_sets, B=64, max_ham=3):
    n = len(sh_sets)
    # TODO: 算每篇指纹 fps[i]=simhash(...)；两两(或同块)比 hamming<=max_ham 则 union；
    #       返回 sorted({find(i)})。小规模可直接两两比。
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
keep_sim = simhash_dedup(sh, B=64, max_ham=6)
assert isinstance(keep_sim, list) and all(isinstance(i,int) for i in keep_sim)
# SimHash 去重应也能把原始语料压缩（簇数 < 原始数）
assert len(keep_sim) < len(sh), 'SimHash 去重后应少于原始文档数'
# 自我一致：同一篇与自己汉明距离 0
assert hamming(simhash(sh[0]), simhash(sh[0])) == 0
print(f'SimHash 去重：{len(sh)} -> {len(keep_sim)} 篇')
print('✅ 练习 3 通过：SimHash + 汉明 + 并查集去重正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_minhash(shingle_set, a, b, p=MERSENNE):
    K = len(a)
    if not shingle_set:
        return np.full(K, p, dtype=np.int64)
    xs = np.array([int(hashlib.sha1(s.encode()).hexdigest()[:15], 16) for s in shingle_set], dtype=np.int64)
    H = (np.outer(xs, a) + b) % p
    return H.min(axis=0)

In [ ]:
# 练习 2 参考答案
def pick_b_r(K, target_tau):
    best = None
    for r in range(1, K + 1):
        if K % r != 0: continue
        b = K // r
        tau = (1.0 / b) ** (1.0 / r)
        d = abs(tau - target_tau)
        if best is None or d < best[0]:
            best = (d, b, r, tau)
    return best[1], best[2], best[3]

In [ ]:
# 练习 3 参考答案
def simhash_dedup(sh_sets, B=64, max_ham=3):
    n = len(sh_sets)
    fps = [simhash(s, B) for s in sh_sets]
    uf = UnionFind(n)
    for i in range(n):
        for j in range(i+1, n):
            if hamming(fps[i], fps[j]) <= max_ham:
                uf.union(i, j)
    return sorted({uf.find(i) for i in range(n)})

---
## 🧪 真实数据胶囊：FineWeb / CommonCrawl 去重算一笔账

用真实语料量级的公开数字（FineWeb / RefinedWeb / CommonCrawl 报告），算去重的复杂度账与收益账：
① 朴素两两去重在 CommonCrawl 单月快照上要多久（不可行）；② MinHash+LSH 的存储与候选量；③ 去重省下的训练算力。

（带 try/except：本环境不联网，直接用内置的真实量级数字。）

In [ ]:
# CommonCrawl / FineWeb 公开量级（约数）
CC_DOCS = 3.0e9          # 单月 CommonCrawl ~30 亿网页(量级)
DEDUP_REMOVE_FRAC = 0.5  # 去重常移除约一半（FineWeb/RefinedWeb 量级）
K_SIG = 128              # MinHash 签名维度

# ① 朴素两两：C(n,2) 次比较
pairs = CC_DOCS * (CC_DOCS - 1) / 2
years_brute = pairs / 1e9 / 3.15e7      # 每秒 1e9 次比较
print(f'① 朴素两两去重: {pairs:.2e} 次比较 -> {years_brute:.2e} 年 (单核, 不可行)')

# ② MinHash 签名存储 + LSH 分桶是 O(n)
sig_bytes = CC_DOCS * K_SIG * 4         # 每签名 K 个 int32
print(f'② MinHash 签名总存储: {sig_bytes/1e12:.2f} TB (O(n), 可分桶并行)')
print(f'   LSH 分桶是 O(n): 每文档算 b 个桶 id，线性扫描即可')

# ③ 去重省下的算力（按 token 正比）
TOKENS_BEFORE = 1.5e13
tokens_saved = TOKENS_BEFORE * DEDUP_REMOVE_FRAC
print(f'③ 去重移除 {DEDUP_REMOVE_FRAC:.0%} -> 省下 {tokens_saved:.2e} token 的存储+分词+训练算力')

assert years_brute > 100, '朴素去重在 CC 量级应是百年级 -> 必须用 LSH'
assert sig_bytes < pairs, 'O(n) 签名存储远小于 O(n^2) 比较量'
print('\n账目结论：O(n²) 不可行；MinHash+LSH 的 O(n) 让 CC 级去重在数小时-数天内可行，')
print('且去重直接省下约一半的下游算力 —— 去重是大规模训练经济可行的前提。')

**🧪 胶囊练习**：实现 `lsh_cost(n, b, r, bytes_per_sig_val=4)`：估算对 `n` 篇文档做 LSH 去重的**签名存储字节数**（n × b × r × bytes_per_sig_val）与**分桶操作数**（n × b，O(n)）。返回 `(storage_bytes, bucket_ops)`。

In [ ]:
def lsh_cost(n, b, r, bytes_per_sig_val=4):
    # TODO: 返回 (n*b*r*bytes_per_sig_val, n*b)
    raise NotImplementedError

In [ ]:
# 自测
storage, ops = lsh_cost(int(3e9), b=20, r=5)
assert storage == int(3e9) * 100 * 4
assert ops == int(3e9) * 20
# 关键：分桶操作是 O(n)，不是 O(n^2)
assert ops < int(3e9) * (int(3e9)-1) / 2
print(f'3e9 文档 LSH: 签名存储 {storage/1e12:.2f} TB，分桶操作 {ops:.2e} 次 (O(n))')
print('✅ 胶囊练习通过：LSH 的存储与操作都是线性的')

In [ ]:
# 📖 胶囊参考答案
def lsh_cost(n, b, r, bytes_per_sig_val=4):
    return n * b * r * bytes_per_sig_val, n * b

---
## 🔧 旁注：真实管线里的去重长什么样

本课的小模拟，在 FineWeb/Dolma 等真实管线里对应：

- **MinHash+LSH** 跑在 Spark/Ray 上：`map` 阶段每文档算签名(易并行)，`shuffle` 阶段按 band 桶 id 重分区(跨 shard 配对)，`reduce` 阶段精筛 + 并查集归簇。
- **(b,r) 调参**：FineWeb 用 K≈112、阈值≈0.8 附近；阈值与下游效果的关系靠消融实验定（这属于 C21 的数据科学范畴）。
- **跨分片合并的并查集**是真实瓶颈：全局并查集需要全局视野，常用分布式并查集或分层合并缓解（Amdahl 在此封顶）。

你在 numpy 里验证过的签名、分桶、归簇逻辑，可几乎一对一搬到 Spark/Ray 的 RDD/Dataset 算子上。

### 小结
- 去重必做：重复导致记忆、虚高困惑度、稀释有效数据（Lee 2021）。
- 精确去重 O(n) 便宜，但抓不住近重复；近重复才是主战场。
- **MinHash** 把集合压成定长签名、无偏估 Jaccard；**LSH banding** 用 S 曲线把 O(n²) 配对降到近 O(n)。
- **SimHash** 是更省的并行路线（64 位指纹 + 汉明）；**并查集**做跨分片归簇。
- 方法论：每个工程版都**对拍**暴力参考、每个机制都**算复杂度账**。

下一站：**模块 02 · 流式加载与分片** —— 去重过的数据有 50 TB，怎么边读边喂、还近似全局打乱？